# Pipeline de Inferencia Batch - Predicción de Incumplimiento Crediticio

## 1. Cargar Librerías

In [ ]:
import pandas as pd
import numpy as np
import pickle

pd.set_option('display.max_columns', None)

## 2. Cargar Modelo Entrenado

In [ ]:
model_path = 'trained_model.pkl'
model = None
try:
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    print(f"Modelo entrenado cargado exitosamente desde: {model_path}")
    print(f"Tipo de modelo cargado: {type(model)}")
except FileNotFoundError:
    print(f"Error: El archivo del modelo no se encontró en la ruta: {model_path}")
except Exception as e:
    print(f"Ocurrió un error al cargar el modelo: {e}")

### Nota sobre Carga de Modelos en Producción:
En un sistema de producción robusto, el modelo no se cargaría directamente desde un archivo `.pkl` local. En su lugar, se gestionaría y versionaría a través de un **Registro de Modelos (Model Registry)** como el que ofrece MLflow. Esto permite un mejor control de versiones, etapas (staging, producción) y facilita la automatización del despliegue de modelos.

## 3. Cargar y Preparar Datos de Inferencia (Batch)

In [ ]:
processed_data_path = 'processed_lending_club_data.csv'
df_full_processed = pd.DataFrame()
X_batch = pd.DataFrame()

try:
    df_full_processed = pd.read_csv(processed_data_path)
    print(f"Datos procesados cargados exitosamente desde: {processed_data_path}. Shape: {df_full_processed.shape}")
    
    # Asegurarse de que la columna 'is_default' (si existe) se elimine para crear el conjunto de características X
    if 'is_default' in df_full_processed.columns:
        X_full = df_full_processed.drop('is_default', axis=1)
        print("Columna 'is_default' eliminada para crear el conjunto de características X_full.")
    else:
        X_full = df_full_processed.copy()
        print("Columna 'is_default' no encontrada en los datos procesados. Se asume que todas las columnas son características.")
    
    # Simular un nuevo batch de datos tomando las primeras 1000 filas
    # En un escenario real, estos serían datos nuevos que llegan al sistema.
    batch_size = 1000
    if len(X_full) >= batch_size:
        X_batch = X_full.head(batch_size).copy() # Usar .copy() para evitar SettingWithCopyWarning más adelante
        print(f"\nBatch de datos para inferencia creado con las primeras {batch_size} filas. Shape de X_batch: {X_batch.shape}")
    else:
        X_batch = X_full.copy()
        print(f"\nEl dataset completo tiene menos de {batch_size} filas. Usando todas las filas para el batch. Shape de X_batch: {X_batch.shape}")
        
    # Verificar que las columnas de X_batch coincidan con las que el modelo espera
    # (esto debería ser así si 'processed_lending_club_data.csv' se generó correctamente)
    if model and hasattr(model, 'feature_name_') and list(X_batch.columns) != list(model.feature_name_):
        print("¡ALERTA! Las columnas de X_batch no coinciden exactamente con las características del modelo entrenado.")
        print(f"Modelo esperaba {len(model.feature_name_)} caracteristicas: {model.feature_name_[:5]}...")
        print(f"Batch tiene {len(X_batch.columns)} caracteristicas: {X_batch.columns.tolist()[:5]}...")
    elif model and hasattr(model, 'n_features_in_') and X_batch.shape[1] != model.n_features_in_:
        print("¡ALERTA! El número de columnas en X_batch no coincide con el número de características esperadas por el modelo.")
        print(f"Modelo esperaba {model.n_features_in_} caracteristicas.")
        print(f"Batch tiene {X_batch.shape[1]} caracteristicas.")
    else:
        print("\nLas características de X_batch parecen coincidir con las esperadas por el modelo.")
        
    display(X_batch.head())

except FileNotFoundError:
    print(f"Error: Datos procesados no encontrados en {processed_data_path}")
except Exception as e:
    print(f"Ocurrió un error al cargar o preparar los datos de batch: {e}")

## 4. Generar Predicciones

In [ ]:
predictions = None
probabilities = None

if model and not X_batch.empty:
    try:
        print("Generando predicciones...")
        predictions = model.predict(X_batch)
        probabilities = model.predict_proba(X_batch)[:, 1] # Probabilidad de la clase '1' (default)
        print("Predicciones generadas exitosamente.")
        
        print(f"\nPrimeras 5 predicciones (0 = No Default, 1 = Default): {predictions[:5]}")
        print(f"Primeras 5 probabilidades de default: {probabilities[:5]}")
    except Exception as e:
        print(f"Error durante la generación de predicciones: {e}")
else:
    print("No se pueden generar predicciones: el modelo no está cargado o X_batch está vacío.")

## 5. Guardar Predicciones

In [ ]:
if predictions is not None and probabilities is not None:
    # Crear un DataFrame para las predicciones
    # Usar el índice original de X_batch como 'record_id' ya que no tenemos otros IDs persistentes.
    df_predictions = pd.DataFrame({
        'record_id': X_batch.index, # El índice de X_batch (que vino de X_full.head())
        'prediction_default': predictions,
        'probability_default': probabilities
    })
    
    output_predictions_path = 'batch_predictions.csv'
    try:
        df_predictions.to_csv(output_predictions_path, index=False)
        print(f"\nPredicciones guardadas exitosamente en: {output_predictions_path}")
        display(df_predictions.head())
    except Exception as e:
        print(f"Error al guardar las predicciones: {e}")
else:
    print("No hay predicciones para guardar.")

## 6. Resumen del Pipeline de Inferencia Batch
1.  **Carga de Librerías**: Se importaron `pandas`, `numpy` y `pickle`.
2.  **Carga del Modelo**: El modelo `LGBMClassifier` previamente entrenado y guardado se cargó desde `trained_model.pkl`. Se incluyó una nota sobre el uso de un Registro de Modelos (como MLflow Model Registry) en entornos productivos para una gestión más robusta.
3.  **Preparación de Datos de Batch**:
    *   Se cargaron los datos procesados desde `processed_lending_club_data.csv` (que contiene las 75 características seleccionadas, escaladas y codificadas, junto con la variable objetivo `is_default`).
    *   Se eliminó la columna `is_default` para obtener el conjunto de características `X_full`.
    *   Se simuló un "nuevo batch" de datos (`X_batch`) tomando las primeras 1000 filas de `X_full`. Esto asegura que `X_batch` tiene exactamente el mismo formato (columnas, orden, preprocesamiento) que los datos con los que se entrenó el modelo.
4.  **Generación de Predicciones**: 
    *   Se utilizó el método `model.predict()` sobre `X_batch` para obtener la clase predicha (0 o 1).
    *   Se utilizó `model.predict_proba()` para obtener la probabilidad de que cada préstamo pertenezca a la clase 'default' (clase 1).
5.  **Guardado de Predicciones**:
    *   Se creó un DataFrame con el índice original de `X_batch` (como `record_id`), la predicción de clase (`prediction_default`) y la probabilidad de default (`probability_default`).
    *   Este DataFrame se guardó en el archivo `batch_predictions.csv`.

Este notebook demuestra un flujo simple para realizar inferencias en batch utilizando un modelo previamente entrenado y datos preprocesados de manera consistente.